# 5. Modelagem com CatBoost

## Objetivo

Treinar e avaliar um modelo de predição de óbito hospitalar por COVID-19 usando **CatBoost** (Categorical Boosting).

## O que é CatBoost?

**CatBoost** é um algoritmo de gradient boosting desenvolvido pelo Yandex que:
- Trata variáveis categóricas nativamente (sem necessidade de codificação)
- Reduz overfitting com técnicas especiais
- É robusto a outliers
- Oferece excelente desempenho em datasets pequenos a médios
- Fornece importância das features

### Diferença entre LightGBM, XGBoost e CatBoost
- **LightGBM**: Mais rápido, melhor para datasets muito grandes
- **XGBoost**: Mais robusto, mais usado em produção
- **CatBoost**: Melhor para dados com muitas categóricas, reduz overfitting

---

## Seção 1: Importações e Carregamento de Dados

In [ ]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json

# CatBoost
from catboost import CatBoostClassifier

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, auc
)

# Configurações
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✓ Bibliotecas importadas com sucesso!")

In [ ]:
# Carregar dados processados
processed_dir = Path("../data/processed")

print("Carregando dados processados...")
X_train = pd.read_csv(processed_dir / "X_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv").squeeze()
y_test = pd.read_csv(processed_dir / "y_test.csv").squeeze()

print(f"✓ Dados carregados com sucesso!")
print(f"\nConjunto de treino: {X_train.shape}")
print(f"Conjunto de teste: {X_test.shape}")

## Seção 2: Treinamento do Modelo CatBoost

In [ ]:
print("="*80)
print("TREINAMENTO DO MODELO CATBOOST")
print("="*80)

# Definir hiperparâmetros
params = {
    'iterations': 100,  # Número de iterações
    'learning_rate': 0.05,  # Taxa de aprendizado
    'depth': 6,  # Profundidade das árvores
    'loss_function': 'Logloss',  # Função de perda para classificação binária
    'eval_metric': 'AUC',  # Métrica de avaliação
    'random_seed': 42,
    'verbose': False,  # Sem mensagens de progresso
    'use_best_model': True,  # Usar melhor modelo
    'early_stopping_rounds': 10,  # Parar se não melhorar
}

print("\nHiperparâmetros:")
for key, value in params.items():
    print(f"  {key}: {value}")

# Treinar modelo
print("\nTreinando modelo...")
model_cat = CatBoostClassifier(**params)

model_cat.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=20
)

print("\n✓ Modelo treinado com sucesso!")

## Seção 3: Avaliação do Modelo

In [ ]:
# Fazer previsões
print("="*80)
print("AVALIAÇÃO DO MODELO CATBOOST")
print("="*80)

# Previsões de probabilidade
y_pred_proba_train = model_cat.predict_proba(X_train)[:, 1]
y_pred_proba_test = model_cat.predict_proba(X_test)[:, 1]

# Previsões binárias
y_pred_train = (y_pred_proba_train >= 0.5).astype(int)
y_pred_test = (y_pred_proba_test >= 0.5).astype(int)

print("\nPrevisões realizadas!")
print(f"Distribuição de previsões no teste:")
print(f"  Sobreviveu (0): {(y_pred_test == 0).sum()}")
print(f"  Óbito (1): {(y_pred_test == 1).sum()}")

In [ ]:
# Calcular métricas
metrics_train = {
    'Acurácia': accuracy_score(y_train, y_pred_train),
    'Precisão': precision_score(y_train, y_pred_train, zero_division=0),
    'Recall': recall_score(y_train, y_pred_train, zero_division=0),
    'F1-Score': f1_score(y_train, y_pred_train, zero_division=0),
    'AUC-ROC': roc_auc_score(y_train, y_pred_proba_train)
}

metrics_test = {
    'Acurácia': accuracy_score(y_test, y_pred_test),
    'Precisão': precision_score(y_test, y_pred_test, zero_division=0),
    'Recall': recall_score(y_test, y_pred_test, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_test, zero_division=0),
    'AUC-ROC': roc_auc_score(y_test, y_pred_proba_test)
}

# Exibir métricas
print("\nMÉTRICAS DE DESEMPENHO - CONJUNTO DE TREINO:")
for metric, value in metrics_train.items():
    print(f"  {metric}: {value:.4f}")

print("\nMÉTRICAS DE DESEMPENHO - CONJUNTO DE TESTE:")
for metric, value in metrics_test.items():
    print(f"  {metric}: {value:.4f}")

# Comparação
print("\nCOMPARAÇÃO TREINO vs TESTE:")
for metric in metrics_train.keys():
    diff = metrics_train[metric] - metrics_test[metric]
    print(f"  {metric}: Treino={metrics_train[metric]:.4f}, Teste={metrics_test[metric]:.4f}, Diferença={diff:.4f}")

In [ ]:
# Matriz de confusão
print("\nMATRIZ DE CONFUSÃO - TESTE:")
cm = confusion_matrix(y_test, y_pred_test)
print(cm)

# Visualizar matriz de confusão
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False, ax=ax,
            xticklabels=['Sobreviveu', 'Óbito'],
            yticklabels=['Sobreviveu', 'Óbito'])
ax.set_xlabel('Previsão')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão - CatBoost (Teste)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/16_catboost_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_test)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(10, 8))
ax.plot(fpr, tpr, color='purple', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Classificador Aleatório')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.set_title('Curva ROC - CatBoost (Teste)', fontsize=12, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/17_catboost_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")
print("\n✓ Gráfico salvo")

## Seção 4: Importância das Features

In [ ]:
# Importância das features
print("="*80)
print("IMPORTÂNCIA DAS FEATURES - CATBOOST")
print("="*80)

# Obter importância
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importância': model_cat.feature_importances_
}).sort_values('Importância', ascending=False)

print("\nTop 15 Features mais importantes:")
print(feature_importance.head(15).to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(12, 8))
top_features = feature_importance.head(15)
ax.barh(range(len(top_features)), top_features['Importância'], color='mediumpurple')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.set_xlabel('Importância')
ax.set_title('Top 15 Features mais Importantes - CatBoost', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../figures/18_catboost_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

## Seção 5: Salvando o Modelo

In [ ]:
# Salvar modelo
results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

# Salvar modelo CatBoost
model_cat.save_model(str(results_dir / "catboost_model.cbm"))

# Salvar métricas
metrics_summary = {
    'algorithm': 'CatBoost',
    'train_metrics': metrics_train,
    'test_metrics': metrics_test,
    'feature_importance': feature_importance.to_dict('list')
}

with open(results_dir / "catboost_metrics.json", 'w') as f:
    json.dump(metrics_summary, f, indent=4)

# Salvar feature importance
feature_importance.to_csv(results_dir / "catboost_feature_importance.csv", index=False)

print("="*80)
print("MODELO E RESULTADOS SALVOS")
print("="*80)

print(f"\n✓ catboost_model.cbm: Modelo treinado")
print(f"✓ catboost_metrics.json: Métricas de desempenho")
print(f"✓ catboost_feature_importance.csv: Importância das features")
print(f"\nLocalização: {results_dir.absolute()}")

## Seção 6: Comparação dos Três Algoritmos

In [ ]:
# Carregar métricas dos outros modelos
with open(results_dir / "lightgbm_metrics.json", 'r') as f:
    lgb_metrics = json.load(f)

with open(results_dir / "xgboost_metrics.json", 'r') as f:
    xgb_metrics = json.load(f)

# Criar tabela comparativa
comparison = pd.DataFrame({
    'Métrica': list(metrics_test.keys()),
    'LightGBM': [lgb_metrics['test_metrics'][m] for m in metrics_test.keys()],
    'XGBoost': [xgb_metrics['test_metrics'][m] for m in metrics_test.keys()],
    'CatBoost': [metrics_test[m] for m in metrics_test.keys()]
})

print("\n" + "="*80)
print("COMPARAÇÃO DOS TRÊS ALGORITMOS (CONJUNTO DE TESTE)")
print("="*80)
print("\n" + comparison.to_string(index=False))

# Visualizar comparação
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(comparison))
width = 0.25

ax.bar(x - width, comparison['LightGBM'], width, label='LightGBM', color='steelblue')
ax.bar(x, comparison['XGBoost'], width, label='XGBoost', color='seagreen')
ax.bar(x + width, comparison['CatBoost'], width, label='CatBoost', color='mediumpurple')

ax.set_ylabel('Score')
ax.set_title('Comparação de Desempenho - Três Algoritmos', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Métrica'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/19_algorithms_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Gráfico salvo")

## Seção 7: Resumo e Conclusões

In [ ]:
print("\n" + "="*80)
print("RESUMO - MODELAGEM COM CATBOOST")
print("="*80)

print(f"""
1. MODELO TREINADO:
   ✓ Algoritmo: CatBoost (Categorical Boosting)
   ✓ Tipo: Classificação binária
   ✓ Variáveis: {X_train.shape[1]}
   ✓ Amostras de treino: {X_train.shape[0]:,}
   ✓ Amostras de teste: {X_test.shape[0]:,}

2. DESEMPENHO NO TESTE:
   ✓ Acurácia: {metrics_test['Acurácia']:.4f} ({metrics_test['Acurácia']*100:.2f}%)
   ✓ Precisão: {metrics_test['Precisão']:.4f}
   ✓ Recall: {metrics_test['Recall']:.4f}
   ✓ F1-Score: {metrics_test['F1-Score']:.4f}
   ✓ AUC-ROC: {metrics_test['AUC-ROC']:.4f}

3. FEATURES MAIS IMPORTANTES:
   Top 3: {', '.join(feature_importance.head(3)['Feature'].tolist())}

4. COMPARAÇÃO DOS TRÊS ALGORITMOS:
   Todos os três algoritmos foram treinados e avaliados.
   Veja a tabela acima para comparação detalhada.

5. PRÓXIMOS PASSOS:
   → Análise SHAP para explicabilidade
   → Análise de justiça algorítmica
   → Transfer learning
   → Conclusões finais
""")

print("="*80)
print("Modelagem com CatBoost concluída! Prossiga para o notebook 06_shap_analysis.ipynb")
print("="*80)